# VLMEvalKit Outputs 결과 확인

In [1]:
import re
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("outputs")

## 1. 전체 결과 로드

In [2]:
score_files = sorted([f for f in OUTPUT_DIR.rglob("*_score.*") if f.suffix in ('.xlsx', '.tsv')])
all_dfs = {}
for f in score_files:
    key = str(f.relative_to(OUTPUT_DIR))
    if f.suffix == '.xlsx':
        df = pd.read_excel(f)
    else:
        df = pd.read_csv(f, sep='\t')
    all_dfs[key] = df
    print(f"{key}: {len(df)} samples, score={df['score'].mean()*100:.2f}%")

Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Instruct_Video-MME_tp1750_mf16_score.tsv: 2700 samples, score=52.78%


## 2. 정확도 요약

In [3]:
summary_rows = []
for key, df in all_dfs.items():
    correct = df["score"].sum()
    total = len(df)
    acc = correct / total * 100
    summary_rows.append({"file": key, "correct": int(correct), "total": total, "accuracy": f"{acc:.2f}%"})

summary_df = pd.DataFrame(summary_rows)
summary_df

,file,correct,total,accuracy
0,Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Inst...,1425,2700,52.78%


In [4]:
## 3. 결과 요약 테이블 (mf를 column으로)

rows = []
for f in score_files:
    rel = f.relative_to(OUTPUT_DIR)
    parts = rel.parts

    simple = "simple" if parts[0].endswith("-simple") else ""
    model = parts[1]

    m = re.search(r'_(Video-MME|MLVU)_tp(\d+)_mf(\d+)', f.stem)
    if not m:
        continue
    dataset = m.group(1)
    mf = int(m.group(3))

    key = str(rel)
    df = all_dfs[key]
    acc = df["score"].mean() * 100

    rows.append({
        "model": model,
        "dataset": dataset,
        "simple": simple,
        "mf": mf,
        "accuracy": round(acc, 2),
        "total": len(df),
    })

result_df = pd.DataFrame(rows)

# mf를 column으로 pivot
pivot = result_df.pivot_table(
    index=["model", "dataset", "simple"],
    columns="mf",
    values="accuracy",
)
pivot.columns = [f"mf={c}" for c in pivot.columns]
pivot = pivot.reset_index()

# total 정보도 추가
pivot_total = result_df.pivot_table(
    index=["model", "dataset", "simple"],
    columns="mf",
    values="total",
)
pivot_total.columns = [f"n={c}" for c in pivot_total.columns]
pivot_total = pivot_total.reset_index()

final = pivot.merge(pivot_total, on=["model", "dataset", "simple"])
acc_cols = sorted([c for c in final.columns if c.startswith("mf=")], key=lambda x: int(x.split("=")[1]))
n_cols = sorted([c for c in final.columns if c.startswith("n=")], key=lambda x: int(x.split("=")[1]))
final = final[["model", "dataset", "simple"] + acc_cols + n_cols]
final

,model,dataset,simple,mf=16,n=16
0,Qwen3-VL-2B-Instruct,Video-MME,,52.78,2700.0
